In [1]:
from nbiatoolkit.models.nbia_responses import (
  Patient,
  PatientList,
  Study,
  StudyList,
  Series,
  SeriesList,
)
from nbiatoolkit import NBIA_BASE_URLS, NBIA_ENDPOINTS, NBIAClient
import requests
from dataclasses import dataclass, field, asdict
from pandas import DataFrame
from typing import Optional, List, Dict, Any, Union, Tuple
from typing import Optional
from pydantic import BaseModel, Field
from enum import Enum

import asyncio
import aiohttp
import aiofiles
from datetime import datetime
import pathlib
import json
import time 


In [2]:
import structlog 
logger = structlog.get_logger()
logger.debug("Starting NBIA Client")
logger.info("Starting NBIA Client")

2024-11-25 13:10:13 [debug    ] Starting NBIA Client          
2024-11-25 13:10:13 [info     ] Starting NBIA Client          


In [3]:
client = NBIAClient()
collections = client.getCollections(return_type="dataframe")
clist = collections.Collection.to_list()


In [4]:
async def fetch_and_write(
	session: aiohttp.ClientSession,
	url: str,
	params: Dict[str, str],
	headers: Optional[Dict[str, str]],
	file_name: pathlib.Path,
	semaphore: asyncio.Semaphore,
	**kwargs: Any
) -> None:
	"""
	Fetches data from the given URL and writes it to a file asynchronously, respecting semaphore limits.

	Parameters
	----------
	session : aiohttp.ClientSession
		An aiohttp session to use for making requests.
	url : str
		The API endpoint URL to fetch data from.
	params : dict
		Parameters to include in the GET request.
	headers : dict, optional
		Headers to include in the GET request.
	file_name : pathlib.Path
		The Path object representing the file to write the content to.
	semaphore : asyncio.Semaphore
		A semaphore to limit the number of concurrent tasks.
	kwargs : Any
		Additional keyword arguments for logging context.

	Returns
	-------
	None
	"""
	async with semaphore:
		thislogger = logger.bind(url=url, **kwargs)
		thislogger.debug("Start fetching")

		async with session.get(url, params=params, headers=headers) as response:
			# Ensure the response is JSON
			if response.headers.get('Content-Type') == 'application/json':
				content = await response.json()
				thislogger.debug("Received JSON response")
			else:
				thislogger.warning("Unexpected Content-Type: %s", response.headers.get('Content-Type'))
				content = None

	# Write the JSON content to the file
	if content is not None:
		async with aiofiles.open(file_name, mode='w') as file:
			await file.write(json.dumps(content, indent=4))  # Pretty print JSON
			thislogger.debug("Written JSON response to file")

async def main(
	query_tuples: List[Tuple[str, Dict[str, str], pathlib.Path]],
	conc_requests: int = 5,
	headers: Optional[Dict[str, str]] = None
) -> None:
	"""
	Main function to fetch data from an API concurrently and write responses to files.
	Uses a semaphore to limit concurrency.

	Parameters
	----------
	query_tuples : list of tuple
		A list of tuples containing URL, request parameters, and output file paths.
	conc_requests : int, optional
		The maximum number of concurrent requests, by default 5.

	Returns
	-------
	None
	"""
	tasks = []
	semaphore = asyncio.Semaphore(conc_requests)  # Limit concurrency

	async with aiohttp.ClientSession() as session:
		for query_tuple in query_tuples:
			url, params, file_name = query_tuple
			task = fetch_and_write(
				session,
				url,
				params,
				headers,
				file_name,
				semaphore,
				collection=params["Collection"]
			)
			tasks.append(task)
		await asyncio.gather(*tasks)



In [5]:
data_path = pathlib.Path("data")

In [ ]:

# Check if there's an existing event loop and run the main function accordingly
try:
	loop = asyncio.get_running_loop()
except RuntimeError:
	loop = None

# URL
CONCURRENT_REQUESTS = 10

# api_url = NBIA_BASE_URLS.NBIA.value + NBIA_ENDPOINTS.GET_PATIENTS.value  # Example API endpoint
endpoints = {
	"Patient": NBIA_ENDPOINTS.GET_PATIENTS.value,
	"Studies": NBIA_ENDPOINTS.GET_STUDIES.value,
	"Series" : NBIA_ENDPOINTS.GET_SERIES.value
}

# Tuples of URL, parameters, and file names
query_tuples = []
for heading, endpoint in endpoints.items():
	url = NBIA_BASE_URLS.NBIA.value + endpoint
	for collection in clist:
		query_tuples.append(
			(
				url, 
				{"Collection": collection }, 
				data_path / "metadata" / collection / f"{heading}List_{collection}.json")
			)

# make all the directories
for _, _, file_name in query_tuples:
	file_name.parent.mkdir(parents=True, exist_ok=True)

start = time.time()
if loop and loop.is_running():
	# If there's an existing event loop, create a task for the main function
	task = asyncio.create_task(main(query_tuples=query_tuples, conc_requests=CONCURRENT_REQUESTS, headers = client.headers))
	await task  # Ensure the task completes before logging the time taken
else:
	# If there's no existing event loop, run the main function
	asyncio.run(main(query_tuples=query_tuples[:10], conc_requests=CONCURRENT_REQUESTS, headers = client.headers))

logger.info("Time taken", time_taken = time.time() - start)

In [6]:
# patient_files = list(data_path.glob("**/PatientList*.json"))

# for i, file in enumerate(patient_files):
#   collection = file.parent.name
#   with file.open(mode="r") as f:
#     data = json.loads(f.read())
#     patient_list = PatientList(items=Patient.from_dicts(data))
collection_paths = list(data_path.glob("metadata/*"))

for collection_path in collection_paths:
  collection = collection_path.name
  if collection != "4D-Lung":
    continue
  series_file = list(collection_path.glob("SeriesList*.json"))[0]
  with series_file.open(mode="r") as f:
    series_list = json.loads(f.read())
  print(len(series_list))
  break

6690
